# Progress Review 2 — Feature Engineering

This notebook creates model-ready airport-hour features from labeled weather data.

Output:

- `s3://lakehouse/gold/training_features_parquet/training_features.parquet`
- `s3://lakehouse/gold/training_features_delta`

In [1]:
import os
import pyspark
from pyspark import SparkContext

print("PySpark package version:", pyspark.__version__)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

existing_sc = SparkContext._active_spark_context

if existing_sc is not None:
    try:
        if existing_sc._jvm is None or existing_sc._jsc is None:
            raise RuntimeError("JVM not available – context is dead")
        existing_sc.stop()
        print("Stopped existing SparkContext.")
    except Exception as e:
        print(f"Existing SparkContext is already dead ({e}); clearing reference.")
    finally:
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None

print("SparkContext cleared – safe to create a new session.")

PySpark package version: 4.0.0
SPARK_HOME: /opt/conda/lib/python3.13/site-packages/pyspark
SparkContext cleared – safe to create a new session.


In [2]:
import pandas as pd
import s3fs

from deltalake import DeltaTable, write_deltalake

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [3]:
spark = (
    SparkSession.builder
    .appName("aviation-pr2-feature-engineering")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "aviation-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4041")
    .config("spark.blockManager.port", "4042")
    .config("spark.ui.port", "4040")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark app ID:", spark.sparkContext.applicationId)

Spark version: 4.0.0
Spark app ID: app-20260425121318-0025


In [4]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT_INTERNAL", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={"endpoint_url": MINIO_ENDPOINT},
)

storage_options = {
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_REGION": AWS_REGION,
    "AWS_ALLOW_HTTP": "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

In [5]:
silver_delta_uri = "s3://lakehouse/silver/weather_labeled_delta"

try:
    dt = DeltaTable(silver_delta_uri, storage_options=storage_options)
    labeled_pdf = dt.to_pandas()
    print("Read Silver from Delta:", silver_delta_uri)

except Exception as exc:
    print("Delta read failed, falling back to Parquet.")
    print("Error:", repr(exc))

    silver_parquet_path = "s3://lakehouse/silver/weather_labeled_parquet/weather_labeled.parquet"
    with fs.open(silver_parquet_path, "rb") as f:
        labeled_pdf = pd.read_parquet(f)

print("Silver shape:", labeled_pdf.shape)
display(labeled_pdf.head())

if labeled_pdf.empty:
    raise RuntimeError("Silver labeled table is empty.")

Read Silver from Delta: s3://lakehouse/silver/weather_labeled_delta
Silver shape: (72, 20)


,event_id,airport,timestamp_utc,event_date,latitude,longitude,temperature_c,wind_speed_kts,precipitation_mm,surface_pressure_pa,cape_j_kg,_kafka_topic,_kafka_partition,_kafka_offset,_ingested_at_utc,_source_object,weather_rule_label,risk_score_proxy,label,label_source
0,JFK_2022-01-01T00:00:00,JFK,2022-01-01T00:00:00Z,2022-01-01,40.75,286.25,9.424164,3.638606,0.005761,101271.765625,12.700439,weather.raw,0,0,2026-04-25T09:40:41.373487+00:00,kafka_weather_events/run_id=20260425T094031Z-d...,0.0,0.158821,0.0,PR2_QUANTILE_FALLBACK
1,JFK_2022-01-01T01:00:00,JFK,2022-01-01T01:00:00Z,2022-01-01,40.75,286.25,9.465088,3.977354,0.010565,101288.007812,21.743164,weather.raw,0,1,2026-04-25T09:40:41.373644+00:00,kafka_weather_events/run_id=20260425T094031Z-d...,0.0,0.181894,0.0,PR2_QUANTILE_FALLBACK
2,JFK_2022-01-01T02:00:00,JFK,2022-01-01T02:00:00Z,2022-01-01,40.75,286.25,9.402130,4.554652,0.012485,101208.484375,23.470459,weather.raw,0,2,2026-04-25T09:40:41.373665+00:00,kafka_weather_events/run_id=20260425T094031Z-d...,0.0,0.206905,0.0,PR2_QUANTILE_FALLBACK
3,JFK_2022-01-01T03:00:00,JFK,2022-01-01T03:00:00Z,2022-01-01,40.75,286.25,9.392700,4.152695,0.001440,101199.937500,17.780762,weather.raw,0,3,2026-04-25T09:40:41.373677+00:00,kafka_weather_events/run_id=20260425T094031Z-d...,0.0,0.184033,0.0,PR2_QUANTILE_FALLBACK
4,JFK_2022-01-01T04:00:00,JFK,2022-01-01T04:00:00Z,2022-01-01,40.75,286.25,9.112518,4.710645,0.000959,101201.640625,14.834229,weather.raw,0,4,2026-04-25T09:40:41.373688+00:00,kafka_weather_events/run_id=20260425T094031Z-d...,0.0,0.203356,0.0,PR2_QUANTILE_FALLBACK


In [6]:
labeled_df = spark.createDataFrame(labeled_pdf)

labeled_df = (
    labeled_df
    .withColumn("timestamp_utc", F.to_timestamp("timestamp_utc"))
    .withColumn("temperature_c", F.col("temperature_c").cast("double"))
    .withColumn("wind_speed_kts", F.col("wind_speed_kts").cast("double"))
    .withColumn("precipitation_mm", F.col("precipitation_mm").cast("double"))
    .withColumn("surface_pressure_pa", F.col("surface_pressure_pa").cast("double"))
    .withColumn("cape_j_kg", F.col("cape_j_kg").cast("double"))
    .withColumn("label", F.col("label").cast("double"))
)

window_3h = (
    Window
    .partitionBy("airport")
    .orderBy(F.col("timestamp_utc").cast("long"))
    .rowsBetween(-2, 0)
)

features_df = (
    labeled_df
    .withColumn("hour_of_day", F.hour("timestamp_utc"))
    .withColumn("day_of_week", F.dayofweek("timestamp_utc"))
    .withColumn("month", F.month("timestamp_utc"))
    .withColumn("wind_speed_max_3h", F.max("wind_speed_kts").over(window_3h))
    .withColumn("wind_speed_avg_3h", F.avg("wind_speed_kts").over(window_3h))
    .withColumn("precipitation_sum_3h", F.sum("precipitation_mm").over(window_3h))
    .withColumn("cape_avg_3h", F.avg("cape_j_kg").over(window_3h))
    .select(
        "event_id",
        "airport",
        "timestamp_utc",
        "temperature_c",
        "wind_speed_kts",
        "wind_speed_max_3h",
        "wind_speed_avg_3h",
        "precipitation_mm",
        "precipitation_sum_3h",
        "surface_pressure_pa",
        "cape_j_kg",
        "cape_avg_3h",
        "hour_of_day",
        "day_of_week",
        "month",
        "label",
        "label_source",
    )
    .dropna()
)

features_df.printSchema()
features_df.show(10, truncate=False)

print("Feature rows:", features_df.count())
features_df.groupBy("label", "label_source").count().show()

root
 |-- event_id: string (nullable = true)
 |-- airport: string (nullable = true)
 |-- timestamp_utc: timestamp (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- wind_speed_kts: double (nullable = true)
 |-- wind_speed_max_3h: double (nullable = true)
 |-- wind_speed_avg_3h: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- precipitation_sum_3h: double (nullable = true)
 |-- surface_pressure_pa: double (nullable = true)
 |-- cape_j_kg: double (nullable = true)
 |-- cape_avg_3h: double (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- label: double (nullable = true)
 |-- label_source: string (nullable = true)

+-----------------------+-------+-------------------+-----------------+-----------------+-----------------+------------------+---------------------+--------------------+-------------------+---------------+------------------+-----------+

In [7]:
features_pdf = features_df.toPandas()

gold_parquet_path = "s3://lakehouse/gold/training_features_parquet/training_features.parquet"

with fs.open(gold_parquet_path, "wb") as f:
    features_pdf.to_parquet(f, index=False)

print("Wrote Gold training features Parquet:", gold_parquet_path)

gold_delta_uri = "s3://lakehouse/gold/training_features_delta"

write_deltalake(
    gold_delta_uri,
    features_pdf,
    mode="overwrite",
    storage_options=storage_options,
)

print("Wrote Gold training features Delta:", gold_delta_uri)
print("Rows:", len(features_pdf))

Wrote Gold training features Parquet: s3://lakehouse/gold/training_features_parquet/training_features.parquet
Wrote Gold training features Delta: s3://lakehouse/gold/training_features_delta
Rows: 72


In [8]:
spark.stop()
print("Spark stopped cleanly.")

Spark stopped cleanly.
